# GRIDPILOT AI — Member 2 — Task 3
## Demand Forecasting Feature Engineering

**Purpose:** Build and unit-test past-only features (lags, rolling stats, weather, calendar) for the demand forecasting model, with explicit leakage-prevention tests.


## 0. Setup

In [ ]:
!pip -q install pandas numpy pyarrow

import numpy as np
import pandas as pd
print("Libraries loaded.")

## 1. Load the canonical dataset produced in Chunk 2

In [ ]:
CANONICAL_PATH = "./data_ingestion_outputs/canonical_demand_dataset.parquet"

df = pd.read_parquet(CANONICAL_PATH)
df = df.sort_values("timestamp").reset_index(drop=True)
print(df.shape)
df.head()

## 2. Lag features (past information only)

Each lag must be a strict shift — never a value from the same or a future row.

In [ ]:
def add_lag_features(data, target_col, freq_minutes, lags_minutes, group_col=None):
    data = data.copy()
    for lag_min in lags_minutes:
        periods = int(lag_min / freq_minutes)
        col_name = f"{target_col}_lag_{lag_min}min"
        if group_col:
            data[col_name] = data.groupby(group_col)[target_col].shift(periods)
        else:
            data[col_name] = data[target_col].shift(periods)
    return data

FREQ_MINUTES = None  # set from Task 1 findings, e.g. 15
ZONE_COL = None       # set if present

assert FREQ_MINUTES is not None, "Set FREQ_MINUTES before running."

df = add_lag_features(
    df,
    target_col="demand" if "demand" in df.columns else df.columns[1],
    freq_minutes=FREQ_MINUTES,
    lags_minutes=[15, 30, 60, 24 * 60],
    group_col=ZONE_COL,
)
df.filter(like="_lag_").head()

## 3. Rolling features (optional, past-only windows)

In [ ]:
DEMAND_COL = "demand" if "demand" in df.columns else df.columns[1]

def add_rolling_features(data, target_col, windows_periods, group_col=None):
    data = data.copy()
    for w in windows_periods:
        mean_col = f"{target_col}_roll_mean_{w}"
        std_col = f"{target_col}_roll_std_{w}"
        shifted = (
            data.groupby(group_col)[target_col].shift(1)
            if group_col else data[target_col].shift(1)
        )
        if group_col:
            data[mean_col] = shifted.groupby(data[group_col]).rolling(w).mean().reset_index(level=0, drop=True)
            data[std_col] = shifted.groupby(data[group_col]).rolling(w).std().reset_index(level=0, drop=True)
        else:
            data[mean_col] = shifted.rolling(w).mean()
            data[std_col] = shifted.rolling(w).std()
    return data

# Only add rolling windows if there is enough history to support them
ROLLING_WINDOWS = [4, 24]  # e.g. 1-hour and 6-hour windows at 15-min frequency
df = add_rolling_features(df, DEMAND_COL, ROLLING_WINDOWS, group_col=ZONE_COL)
df.filter(like="_roll_").head()

## 4. Weather / solar / wind features

Only include columns that were confirmed to exist in Task 1.

In [ ]:
WEATHER_COLS = {
    "temperature": None,
    "humidity": None,
    "cloud_cover": None,
    "wind_speed": None,
    "solar_radiation": None,
}

for logical_name, col in WEATHER_COLS.items():
    if col is not None and col in df.columns:
        print(f"Including weather feature: {logical_name} -> {col}")
    else:
        print(f"Skipping {logical_name}: column not confirmed/available.")

## 5. Calendar features

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["month"] = df["timestamp"].dt.month

def month_to_season(m):
    if m in (12, 1, 2):
        return "winter"
    if m in (3, 4, 5):
        return "spring"
    if m in (6, 7, 8):
        return "summer"
    return "autumn"

df["season"] = df["month"].apply(month_to_season)

# Holiday flag placeholder -- wire up an actual holiday calendar for the target region
HOLIDAY_DATES = set()  # e.g. {"2024-01-01", "2024-12-25"}
df["is_holiday"] = df["timestamp"].dt.date.astype(str).isin(HOLIDAY_DATES).astype(int)

df[["timestamp", "hour", "day_of_week", "is_weekend", "season", "is_holiday"]].head()

## 6. Leakage prevention tests

Prove that no feature at row *t* uses information from timestamp > t.

In [ ]:
def test_lag_features_no_leakage(data, target_col, lag_col, periods):
    """A lag feature at row i must equal target_col at row i-periods (or NaN)."""
    shifted = data[target_col].shift(periods)
    mismatches = (
        data[lag_col].notna()
        & shifted.notna()
        & (data[lag_col] != shifted)
    )
    assert mismatches.sum() == 0, f"Leakage detected in {lag_col}: {mismatches.sum()} mismatches"
    print(f"PASS: {lag_col} contains only past values.")

test_lag_features_no_leakage(df, DEMAND_COL, f"{DEMAND_COL}_lag_15min", int(15 / FREQ_MINUTES))
test_lag_features_no_leakage(df, DEMAND_COL, f"{DEMAND_COL}_lag_60min", int(60 / FREQ_MINUTES))

def test_rolling_uses_shifted_data(data, mean_col):
    """Rolling mean columns must be NaN for the first rows (no history) -- a cheap smoke test."""
    n_leading_nan = data[mean_col].head(5).isna().sum()
    assert n_leading_nan > 0, f"{mean_col} looks suspicious -- expected NaNs at the very start"
    print(f"PASS: {mean_col} has expected leading NaNs (no future leakage into warm-up rows).")

test_rolling_uses_shifted_data(df, f"{DEMAND_COL}_roll_mean_{ROLLING_WINDOWS[0]}")

print("\nAll leakage tests passed.")

## 7. Save the feature-engineered dataset

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("./feature_engineering_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

feature_path = OUTPUT_DIR / "demand_features.parquet"
df.to_parquet(feature_path, index=False)
print("Saved:", feature_path, "shape:", df.shape)

## Next step

Port this feature logic into `services/forecasting/**` / `ml/experiments/demand/**` as a reusable, versioned feature pipeline before Chunk 5 (LightGBM).